# Imports and Parameter Definitions

In [22]:
using Pkg

In [2]:
using JuMP
using HiGHS
#using CSV
using DataFrames
using LinearAlgebra

In [25]:
struct ModelData
    T::Vector{Int} #time periods
    G::Vector{Int} #conventional units
    R::Vector{Int} #wind units
    N::Vector{Int} #nodes
    B::Vector{Int} #batteries
    Ω::Vector{Int}      # intra-day scenarios
    Σ::Vector{Int}      # real-time scenarios
    G_n::Dict{Int,Vector{Int}} #conventional units at node n
    R_n::Dict{Int,Vector{Int}} #wind units at node n
    B_n::Dict{Int,Vector{Int}} #batteries at node n
    neighbors::Dict{Int,Vector{Int}} #node connections
    E0::Dict{Int,Float64} #starting charge battery
    
    # Probabilities
    πΩ::Dict{Int,Float64} #intra scenario probs
    πΣ::Dict{Tuple{Int,Int},Float64} #real time wind probs
    
    # Costs
    C::Dict{Int,Float64} # conv unit marginal cost
    C_spill::Float64 # cost of wind spill real time
    VOLL::Float64 # cost of load shedding real time
    C_deg::Dict{Int,Float64} # battery degredation cost 
    
    # Generator limits
    Pmin::Dict{Int,Float64} # conv unit min prod
    Pmax::Dict{Int,Float64} # conv unit max prod
    RU::Dict{Int,Float64} # ramp up limit of conv prod between consequtive t
    RD::Dict{Int,Float64} # ramp down limit of conv prod between consequtive t
    
    # Renewable forecasts
    W_DA::Dict{Tuple{Int,Int},Float64} # day ahead (expected value of intra scenarios)
    W_ID::Dict{Tuple{Int,Int,Int},Float64} # intra day scenarios
    W_RT::Dict{Tuple{Int,Int,Int,Int},Float64} # real time wind
    
    # Demand
    D::Dict{Tuple{Int,Int},Float64} # demand at each node
    
    # Network
    Fmax::Dict{Tuple{Int,Int},Float64} # line caps
    Bline::Dict{Tuple{Int,Int},Float64} # line susceptability
    
    # Battery
    ηch::Dict{Int,Float64} # charging efficiency
    ηdis::Dict{Int,Float64} # discharging efficiency
    Emin::Dict{Int,Float64} # min charge
    Emax::Dict{Int,Float64} # max charge
    Δt::Float64 # 
end


In [26]:
# -------------------------
# SETS
# -------------------------
T  = [1,2,3] # time
G = [1,2,3,4] # conv units
R  = [1] # wind units
N = [1,2,3,4,5] #nodes
B  = [1] # batteries
Ω  = [1,2] #intra scenarios
Σ = [1,2,3] # real time scenarios

# -------------------------
# SCENARIO PROBABILITIES
# -------------------------
πΩ = Dict(1 => 0.396, #intra
          2 => 0.604)

πΣ = Dict(
    (1,1)=>1/5, (1,2)=>3/5, (1,3)=>1/5, #real time
    (2,1)=>1/5, (2,2)=>3/5, (2,3)=>1/5
)

# -------------------------
# COSTS
# -------------------------
C   = Dict(1 => 30.0, #conv marginal cost
              2 => 25.0,
              3 => 40.0,
              4 => 70.0)
C_spill = 5.0 # wind spillage cost
VOLL    = 1000.0 # cost of lost demand
C_deg   = Dict(1 => 1.0) #degredation cost factor

# -------------------------
# GENERATOR LIMITS
# -------------------------
Pmin = Dict(1 => 0.0,
            2 => 0.0,
            3 => 0.0,
            4 => 0.0)
Pmax = Dict(1=>100.0, 2=>80.0, 3=>70.0, 4=>60.0)  
RU = Dict(1=>20.0, 2=>20.0, 3=>20.0, 4=>20.0) #ramp up conv units
RD = Dict(1=>20.0, 2=>20.0, 3=>20.0, 4=>20.0) #ramp down units

# -------------------------
# WIND FORECASTS
# -------------------------
W_DA = Dict((1,1)=>35,
            (1,2)=>36,
            (1,3)=>33)

W_ID = Dict((1,1,1)=>88.9, (1,1,2)=>0.5,
            (1,2,1)=>92.5, (1,2,2)=>0.2,
            (1,3,1)=>84.5, (1,3,2)=>0.6)

W_RT = Dict(
 (1,1,1,1)=>44.2, (1,1,1,2)=>50.8, (1,1,1,3)=>140.4,
 (1,1,2,1)=>0.0, (1,1,2,2)=>0.5, (1,1,2,3)=>1.0,
 (1,2,1,1)=>33.88, (1,2,1,2)=>86.8, (1,2,1,3)=>179.8,
 (1,2,2,1)=>0.0, (1,2,2,2)=>0.2, (1,2,2,3)=>1.2,
 (1,3,1,1)=>36.6, (1,3,1,2)=>103.5, (1,3,1,3)=>178.4,
 (1,3,2,1)=>0.1, (1,3,2,2)=>1.4, (1,3,2,3)=>3.0
)

# -------------------------
# DEMAND (n,t)
# -------------------------
D = Dict((1,1)=>100.0, (2,1)=>20.0, (3,1)=>80.0, (4,1)=>70.0, (5,1)=>0,
         (1,2)=>120.0, (2,2)=>50.0, (3,2)=>95.0, (4,2)=>85.0, (5,2)=>0,
         (1,3)=>110.0, (2,3)=>20.0, (3,3)=>85.0, (4,3)=>75.0, (5,3)=>0)


# -------------------------
# NETWORK (n,r), where r the set of connected nodes to n 
# -------------------------
Fmax = Dict((1,2)=>120.0, (1,4)=>120.0, (2,1)=>120.0, (2,3)=>120.0, (2,5)=>120.0, (3,2)=>120.0, (4,1)=>120.0, (4,5)=>120.0, (5,2)=>120.0, (5,4)=>120.0, (2,1)=>120.0, (4,1)=>120.0, (1,2)=>120.0, (3,2)=>120.0, (5,2)=>120.0, (2,3)=>120.0, (1,4)=>120.0, (5,4)=>120.0, (2,5)=>120.0, (4,5)=>120.0)
Bline = Dict((1,2)=>6.67, (1,4)=>6.67, (2,1)=>6.67, (2,3)=>6.67, (2,5)=>6.67, (3,2)=>6.67, (4,1)=>6.67, (4,5)=>6.67, (5,2)=>6.67, (5,4)=>6.67, (2,1)=>6.67, (4,1)=>6.67, (1,2)=>6.67, (3,2)=>6.67, (5,2)=>6.67, (2,3)=>6.67, (1,4)=>6.67, (5,4)=>6.67, (2,5)=>6.67, (4,5)=>6.67)

neighbors = Dict(
    1 => [2,4], # node 1 has lines to node 2 and 4
    2 => [1,3,5],
    3 => [2],
    4 => [1,5],
    5 => [2,4]
)

# -------------------------
# NODE ASSIGNMENTS
# -------------------------
G_n = Dict(1=>[1,2], 2=>Int[], 3=>[3], 4=>[4], 5=>Int[]) #assignment of each node to conv units
R_n = Dict(1=>Int[], 2=>Int[], 3=>Int[], 4=>Int[], 5=>[1]) #assignment of each node to wind units
B_n = Dict(1=>Int[], 2=>[1], 3=>Int[], 4=>Int[], 5=>Int[]) #assignment of each node to batteries

# -------------------------
# BATTERY
# -------------------------
ηch  = Dict(1=>0.95)
ηdis = Dict(1=>0.95)
Emin = Dict(1=>0.0)
Emax = Dict(1=>100.0)
E0   = Dict(1=>50.0)
Δt   = 1.0

1.0

In [28]:
data = ModelData(
    T,G,R,N,B,Ω,Σ,G_n,R_n,B_n,neighbors,E0,
    πΩ,πΣ,
    C,C_spill,VOLL,C_deg,
    Pmin,Pmax,RU,RD,
    W_DA,W_ID,W_RT,
    D,
    Fmax,Bline,
    ηch,ηdis,Emin,Emax,
    Δt
)

ModelData([1, 2, 3], [1, 2, 3, 4], [1], [1, 2, 3, 4, 5], [1], [1, 2], [1, 2, 3], Dict(5 => [], 4 => [4], 2 => [], 3 => [3], 1 => [1, 2]), Dict(5 => [1], 4 => [], 2 => [], 3 => [], 1 => []), Dict(5 => [], 4 => [], 2 => [1], 3 => [], 1 => []), Dict(5 => [2, 4], 4 => [1, 5], 2 => [1, 3, 5], 3 => [2], 1 => [2, 4]), Dict(1 => 50.0), Dict(2 => 0.604, 1 => 0.396), Dict((1, 2) => 0.6, (1, 1) => 0.2, (1, 3) => 0.2, (2, 2) => 0.6, (2, 1) => 0.2, (2, 3) => 0.2), Dict(4 => 70.0, 2 => 25.0, 3 => 40.0, 1 => 30.0), 5.0, 1000.0, Dict(1 => 1.0), Dict(4 => 0.0, 2 => 0.0, 3 => 0.0, 1 => 0.0), Dict(4 => 60.0, 2 => 80.0, 3 => 70.0, 1 => 100.0), Dict(4 => 20.0, 2 => 20.0, 3 => 20.0, 1 => 20.0), Dict(4 => 20.0, 2 => 20.0, 3 => 20.0, 1 => 20.0), Dict((1, 2) => 36.0, (1, 1) => 35.0, (1, 3) => 33.0), Dict((1, 2, 1) => 92.5, (1, 1, 1) => 88.9, (1, 3, 1) => 84.5, (1, 2, 2) => 0.2, (1, 1, 2) => 0.5, (1, 3, 2) => 0.6), Dict((1, 3, 2, 1) => 0.1, (1, 1, 1, 3) => 140.4, (1, 2, 1, 2) => 86.8, (1, 2, 1, 1) => 33.88, (1,

# Model Definition

In [36]:
function build_model(data::ModelData)

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    T,G,R,N,B = data.T, data.G, data.R, data.N, data.B
    Ω,Σ = data.Ω, data.Σ

    # -------------------------
    # FIRST STAGE
    # -------------------------
    @variable(model, P[i in G, t in T]) # G conv generators. Day-ahead scheduled production of unit i at time t.
    @variable(model, W[q in R, t in T] >= 0) # R renewables (wind). Day-ahead scheduled renewable power production of unit q at time t.
    @variable(model, θ[n in N, t in T]) # Angles.
    @variable(model, Pch[b in B, t in T] >= 0) # B batteries. Day-ahead charge or discharge of battery b at time t.
    @variable(model, Pdis[b in B, t in T] >= 0) 
    @variable(model, E[b in B, t in T]) # Battery state-of-charge at day-ahead.

    # -------------------------
    # SECOND STAGE
    # -------------------------
    @variable(model, ΔP_up[i in G, t in T, ω in Ω] >= 0) # G conv generators, w intra scenarios. Upward redispatch of unit i time t in scenario w
    @variable(model, ΔP_dn[i in G, t in T, ω in Ω] >= 0) # G conv generators, w intra scenarios. downward redispatch of unit i time t in scenario w
    @variable(model, ΔW_up[q in R, t in T, ω in Ω] >= 0) # Upward and downward renewable adjustment of unit q at time t in scenario w.
    @variable(model, ΔW_dn[q in R, t in T, ω in Ω] >= 0)
    @variable(model, ΔPch_ID[b in B, t in T, ω in Ω] >= 0) # B batteries. Intra-day charge/discharge adjustment.
    @variable(model, ΔPdis_ID[b in B, t in T, ω in Ω] >= 0)
    @variable(model, θ_ID[n in N, t in T, ω in Ω])
    @variable(model, E_ID[b in B, t in T, ω in Ω]) # Battery state-of-charge at intra-day.

    # -------------------------
    # THIRD STAGE
    # -------------------------
    @variable(model, rU[i in G, t in T, ω in Ω, σ in Σ] >= 0) # Upward and downward regulation for conv unit i at time t inscenario σ given scenario w.
    @variable(model, rD[i in G, t in T, ω in Ω, σ in Σ] >= 0)
    @variable(model, spill[q in R, t in T, ω in Ω, σ in Σ] >= 0) # renewable energy spillage for renewable unit q at time t inscenario σ given scenario w.
    @variable(model, shed[n in N, t in T, ω in Ω, σ in Σ] >= 0) # Load shedding for demand at node n (of N)
    @variable(model, ΔPch_RT[b in B, t in T, ω in Ω, σ in Σ] >= 0) # Real-time charge/discharge adjustment.
    @variable(model, ΔPdis_RT[b in B, t in T, ω in Ω, σ in Σ] >= 0) # Real-time charge/discharge adjustment.
    @variable(model, θ_RT[n in N, t in T, ω in Ω, σ in Σ])
    @variable(model, E_RT[b in B, t in T, ω in Ω, σ in Σ]) # Battery state-of-charge at real-time.
    @variable(model, E_RT_exp[b in B, t in T]) # Expected (weighted) real-time SoC at the end of period t.

    # -------------------------
    # OBJECTIVE: system-operator (realized production) form
    # -------------------------
    @objective(model, Min,
        # outer expectation: intra-day scenarios ω
        sum( data.πΩ[ω] * (
            # inner expectation: conditional real-time scenarios σ | ω
            sum( data.πΣ[(ω,σ)] * (
                    # realized thermal production cost (charged once, on realized output)
                    sum( data.C[i] * ( 
                            P[i,t] 
                            + ΔP_up[i,t,ω] 
                            - ΔP_dn[i,t,ω] 
                            + rU[i,t,ω,σ] 
                            - rD[i,t,ω,σ]
                        )
                        for i in G, t in T )
    
                    # wind spillage
                    + sum( data.C_spill * spill[q,t,ω,σ] for q in R, t in T )
    
                    # load shedding
                    + sum( data.VOLL * shed[n,t,ω,σ] for n in N, t in T )
                ) for σ in Σ )   # end inner sum over σ
        ) for ω in Ω )        # end outer sum over ω
    )

    # -------------------------
    # DAY-AHEAD CONSTRAINTS
    # -------------------------

    @constraint(model, [i in G, t in T],
        data.Pmin[i] <= P[i,t] <= data.Pmax[i]) # conv power caps for day ahead schedule

    @constraint(model, [i in G, t in T[2:end]],
    -data.RD[i] <= P[i,t] - P[i,t-1] <= data.RU[i])

    @constraint(model, [q in R, t in T],
        W[q,t] <= data.W_DA[(q,t)]) # wind power cap for schedule (cap is da forcast)

    # Power balance
    @constraint(model, [n in N, t in T],
        sum(P[i,t] for i in data.G_n[n]) + # the production of conv units
        sum(W[q,t] for q in data.R_n[n]) + # the wind prod
        sum(Pdis[b,t] - Pch[b,t] for b in data.B_n[n]) - # the charging or discharging amount of battery
        data.D[(n,t)] #the demand
        ==
        sum(data.Bline[(n,r)] *
            (θ[n,t] - θ[r,t])
            for r in data.neighbors[n]) # the line flows
    )

    # Line / Transmission limits
    @constraint(model, [n in N, r in data.neighbors[n], t in T],
        -data.Fmax[(n,r)]
        <= data.Bline[(n,r)] * (θ[n,t] - θ[r,t])
        <= data.Fmax[(n,r)]
    )

    # SoC dynamics (DA) for t=1
    @constraint(model, [b in B],
        E[b,first(T)] ==
        data.E0[b] +
        data.ηch[b]*Pch[b,first(T)]*data.Δt -
        (1/data.ηdis[b])*Pdis[b,first(T)]*data.Δt
    )
     # SoC dynamics (DA) for t>1
    @constraint(model, [b in B, t in T[2:end]],
        E[b,t] ==
        E_RT_exp[b,t-1] + #expected SoC from end of t-1
        data.ηch[b]*Pch[b,t]*data.Δt -
        (1/data.ηdis[b])*Pdis[b,t]*data.Δt
    )
    # SOC caps
    @constraint(model, [b in B, t in T],
        data.Emin[b] <= E[b,t] <= data.Emax[b])

    @constraint(model, [t in T],
    θ[1,t] == 0)


    # -------------------------
    # INTRA-DAY CONSTRAINTS
    # -------------------------

    # Power balance
    @constraint(model, [n in N, t in T, ω in Ω],
        sum(ΔP_up[i,t,ω] - ΔP_dn[i,t,ω]
            for i in data.G_n[n])
        +
        sum(ΔW_up[q,t,ω] - ΔW_dn[q,t,ω] for q in data.R_n[n])
        +
        sum(ΔPdis_ID[b,t,ω] - ΔPch_ID[b,t,ω]
            for b in data.B_n[n])
        ==
        sum(data.Bline[(n,r)] *
            (θ_ID[n,t,ω] - θ_ID[r,t,ω] - θ[n,t] + θ[r,t])
            for r in data.neighbors[n])
    )
    
    # Redispatch feasibility for conv units
    @constraint(model, [i in G, t in T, ω in Ω],
        data.Pmin[i]
        <= P[i,t] + ΔP_up[i,t,ω] - ΔP_dn[i,t,ω]
        <= data.Pmax[i]
    )

    @constraint(model, [i in G, t in T[2:end], ω in Ω],
        -data.RD[i]
        <= (P[i,t] + ΔP_up[i,t,ω] - ΔP_dn[i,t,ω]) - (P[i,t-1] + ΔP_up[i,t-1,ω] - ΔP_dn[i,t-1,ω])
        <= data.RU[i])

    # Renewable forecast update
    @constraint(model, [q in R, t in T, ω in Ω],
        0
        <= W[q,t] + ΔW_up[q,t,ω] - ΔW_dn[q,t,ω]
        <= data.W_ID[(q,t,ω)]
    )
    # SOC update
    @constraint(model, [b in B, t in T, ω in Ω],
        E_ID[b,t,ω] == E[b,t] +
        data.ηch[b]*ΔPch_ID[b,t,ω]*data.Δt -
        (1/data.ηdis[b])*ΔPdis_ID[b,t,ω]*data.Δt)

    # SOC caps
    @constraint(model, [b in B, t in T, ω in Ω],
        data.Emin[b] <= E_ID[b,t,ω] <= data.Emax[b])

    # Line / Transmission limits
    @constraint(model, [n in N, r in data.neighbors[n], t in T, ω in Ω],
        -data.Fmax[(n,r)]
        <= data.Bline[(n,r)] * (θ_ID[n,t,ω] - θ_ID[r,t,ω])
        <= data.Fmax[(n,r)]
    )

    @constraint(model, [t in T, ω in Ω],
    θ_ID[1,t,ω] == 0)

    # -------------------------
    # REAL-TIME CONSTRAINTS
    # -------------------------

        # balance
    @constraint(model, RT_balance[n in N, t in T, ω in Ω, σ in Σ],
        sum(rU[i,t,ω,σ] - rD[i,t,ω,σ]
            for i in data.G_n[n])
        +
        sum(data.W_RT[(q,t,ω,σ)] - (W[q,t] + ΔW_up[q,t,ω] - ΔW_dn[q,t,ω]) - spill[q,t,ω,σ]
            for q in data.R_n[n])
        +
        sum(ΔPdis_RT[b,t,ω,σ] - ΔPch_RT[b,t,ω,σ]
            for b in data.B_n[n])
        +
        shed[n,t,ω,σ]
        ==
        sum(data.Bline[(n,r)] *
            (θ_RT[n,t,ω,σ] - θ_RT[r,t,ω,σ] - θ_ID[n,t,ω] + θ_ID[r,t,ω])
            for r in data.neighbors[n])
    )

        # Redispatch feasibility for conv units
    @constraint(model, [i in G, t in T, ω in Ω, σ in Σ],
        data.Pmin[i]
        <= P[i,t] + ΔP_up[i,t,ω] - ΔP_dn[i,t,ω] + rU[i,t,ω,σ] - rD[i,t,ω,σ]
        <= data.Pmax[i]
    )

    # ramping limits conv units
    @constraint(model, [i in G, t in T[2:end], ω in Ω, σ in Σ],
        -data.RD[i]
        <= (P[i,t] + ΔP_up[i,t,ω] - ΔP_dn[i,t,ω] + rU[i,t,ω,σ] - rD[i,t,ω,σ]) - (P[i,t-1] + ΔP_up[i,t-1,ω] - ΔP_dn[i,t-1,ω] + rU[i,t-1,ω,σ] - rD[i,t-1,ω,σ])
        <= data.RU[i])

    # reserve usage cap
    @constraint(model, [i in G, t in T, ω in Ω, σ in Σ],
        rU[i,t,ω,σ] <= data.RU[i])

    # reserve usage cap
    @constraint(model, [i in G, t in T, ω in Ω, σ in Σ],
        rD[i,t,ω,σ] <= data.RD[i])

    # shedding
    @constraint(model, [n in N, t in T, ω in Ω, σ in Σ],
        shed[n,t,ω,σ] >= 0)

     #SOC update
    @constraint(model, SOC_RT[b in B, t in T, ω in Ω, σ in Σ],
        E_RT[b,t,ω,σ] == E_ID[b,t,ω] +
        data.ηch[b]*ΔPch_RT[b,t,ω,σ]*data.Δt -
        (1/data.ηdis[b])*ΔPdis_RT[b,t,ω,σ]*data.Δt)
    
    # battery state limit
    @constraint(model, [b in B, t in T, ω in Ω, σ in Σ],
        data.Emin[b] <= E_RT[b,t,ω,σ] <= data.Emax[b])

    # Line / Transmission limits
    @constraint(model, [n in N, r in data.neighbors[n], t in T, ω in Ω, σ in Σ],
        -data.Fmax[(n,r)]
        <= data.Bline[(n,r)] * (θ_RT[n,t,ω,σ] - θ_RT[r,t,ω,σ])
        <= data.Fmax[(n,r)]
    )

    @constraint(model, [t in T, ω in Ω, σ in Σ],
    θ_RT[1,t,ω,σ] == 0)


    # -------------------------
    # EXPECTED SOC
    # -------------------------

    @constraint(model, SOC_exp_link[b in B, t in T],
        E_RT_exp[b,t] ==
        sum(data.πΩ[ω] *
            sum(data.πΣ[(ω,σ)] *
                E_RT[b,t,ω,σ]
                for σ in Σ)
            for ω in Ω)
    )

    return model
end


build_model (generic function with 2 methods)

# Model Run

In [37]:
model = build_model(data)

A JuMP Model
├ solver: HiGHS
├ objective_sense: MIN_SENSE
│ └ objective_function_type: AffExpr
├ num_variables: 546
├ num_constraints: 1281
│ ├ AffExpr in MOI.EqualTo{Float64}: 192
│ ├ AffExpr in MOI.GreaterThan{Float64}: 90
│ ├ AffExpr in MOI.LessThan{Float64}: 147
│ ├ AffExpr in MOI.Interval{Float64}: 483
│ └ VariableRef in MOI.GreaterThan{Float64}: 369
└ Names registered in the model
  └ :E, :E_ID, :E_RT, :E_RT_exp, :P, :Pch, :Pdis, :RT_balance, :SOC_RT, :SOC_exp_link, :W, :rD, :rU, :shed, :spill, :ΔP_dn, :ΔP_up, :ΔPch_ID, :ΔPch_RT, :ΔPdis_ID, :ΔPdis_RT, :ΔW_dn, :ΔW_up, :θ, :θ_ID, :θ_RT

In [38]:
optimize!(model)

# Check
status = termination_status(model)
println("Status: ", status)

# Total expected cost
if status == MOI.OPTIMAL || status == MOI.LOCALLY_SOLVED
    total_cost = objective_value(model)
    println("Total Expected Cost: \$", round(total_cost, digits=2))
else
    println("Nonoptimal")
end

Status: OPTIMAL
Total Expected Cost: $25559.46


# Decision Variable results

In [46]:
# Day-ahead generator schedule
println("P")
P_out = value.(model[:P]) # Extracts the DenseAxisArray of values
display(P_out)

println("ΔP_dn")
spill = value.(model[:ΔP_dn])
display(spill)

println("ΔP_up")
shed = value.(model[:ΔP_up])
display(shed)

println("rD")
rD = value.(model[:rD])
display(rD)

println("rU")
rU = value.(model[:rU])
display(rU)

P


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
And data, a 4×3 Matrix{Float64}:
 80.0  100.0  95.6929
 65.0   80.0  60.0
 50.0   70.0  50.0
 40.0   60.0  40.0

ΔP_dn


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 4×3×2 Array{Float64, 3}:
[:, :, 1] =
  0.0   0.0     15.6929
  0.0  -0.0     -0.0
  0.0  15.8071  15.8071
 20.0  40.0     20.0

[:, :, 2] =
 0.0       -0.0       0.0
 4.0       -0.0       0.0
 0.0       -0.0       0.0
 0.988138   0.988138  0.0

ΔP_up


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 4×3×2 Array{Float64, 3}:
[:, :, 1] =
 12.2316  -0.0  0.0
 15.0      0.0  0.0
 -0.0      0.0  0.0
  0.0      0.0  0.0

[:, :, 2] =
 20.0  0.0   4.30708
  0.0  0.0   3.0
 20.0  0.0  20.0
  0.0  0.0   5.69292

rD


4-dimensional DenseAxisArray{Float64,4,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
    Dimension 4, [1, 2, 3]
And data, a 4×3×2×3 Array{Float64, 4}:
[:, :, 1, 1] =
  0.0  -0.0      0.0
 -0.0  -0.0      0.0
  0.0   0.0      0.0
 20.0  13.6681  12.1

[:, :, 2, 1] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0   0.0
 18.8  18.8  17.1

[:, :, 1, 2] =
  0.0  -0.0       0.0
 -0.0  -0.0       0.0
  0.0   4.19292   4.19292
 20.0  20.0      20.0

[:, :, 2, 2] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0  -0.0
 19.0  19.0  18.4

[:, :, 1, 3] =
 20.0  20.0  20.0
 -0.0  20.0  20.0
 20.0  20.0  20.0
 20.0  20.0  20.0

[:, :, 2, 3] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0  -0.0
 20.0  20.0  20.0

rU


4-dimensional DenseAxisArray{Float64,4,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
    Dimension 4, [1, 2, 3]
And data, a 4×3×2×3 Array{Float64, 4}:
[:, :, 1, 1] =
  7.76842   0.0     20.0
  0.0       0.0     20.0
 20.0      15.8071  20.0
  0.0       0.0      0.0

[:, :, 2, 1] =
 -0.0  0.0   0.0
 19.0  0.0  17.0
  0.0  0.0  -0.0
  0.0  0.0   0.0

[:, :, 1, 2] =
  7.76842  0.0  -0.0
  0.0      0.0   5.19292
 20.0      0.0   0.0
  0.0      0.0   0.0

[:, :, 2, 2] =
 -0.0  0.0   0.0
 19.0  0.0  17.0
  0.0  0.0   0.0
  0.0  0.0   0.0

[:, :, 1, 3] =
 0.0  -0.0   0.0
 0.0   0.0   0.0
 0.0   0.0   0.0
 0.0   0.0  -0.0

[:, :, 2, 3] =
 -0.0   0.0   0.0
 19.0   0.0  17.0
  0.0   0.0   0.0
  0.0  -0.0   0.0

In [47]:
# Day-ahead generator schedule
println("W")
W = value.(model[:W]) # Extracts the DenseAxisArray of values
display(W)

println("ΔW_dn")
ΔW_dn = value.(model[:ΔW_dn]) # Extracts the DenseAxisArray of values
display(ΔW_dn)

println("ΔW_up")
ΔW_up = value.(model[:ΔW_up]) # Extracts the DenseAxisArray of values
display(ΔW_up)

println("spill")
spill = value.(model[:spill]) # Extracts the DenseAxisArray of values
display(spill)

W


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
And data, a 1×3 Matrix{Float64}:
 35.0  36.0  33.0

ΔW_dn


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 1×3×2 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0

[:, :, 2] =
 35.0  36.0  33.0

ΔW_up


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 1×3×2 Array{Float64, 3}:
[:, :, 1] =
 16.9684  55.8071  51.5

[:, :, 2] =
 0.0  0.0  0.0

spill


4-dimensional DenseAxisArray{Float64,4,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
    Dimension 4, [1, 2, 3]
And data, a 1×3×2×3 Array{Float64, 4}:
[:, :, 1, 1] =
 0.0  0.0  0.0

[:, :, 2, 1] =
 0.0  0.0  0.0

[:, :, 1, 2] =
 0.0  0.0  0.0

[:, :, 2, 2] =
 0.0  0.0  0.0

[:, :, 1, 3] =
 0.0  0.0  0.0

[:, :, 2, 3] =
 0.0  0.0  0.0

In [21]:
println("E")
E = value.(model[:E]) # Extracts the DenseAxisArray of values
display(E)

println("E_ID")
E_ID = value.(model[:E_ID]) # Extracts the DenseAxisArray of values
display(E_ID)

println("E_RT")
E_RT = value.(model[:E_RT]) # Extracts the DenseAxisArray of values
display(E_RT)

println("E_RT_exp")
E_RT_exp = value.(model[:E_RT_exp]) # Extracts the DenseAxisArray of values
display(E_RT_exp)


println("Pch")
Pch = value.(model[:Pch]) # Extracts the DenseAxisArray of values
display(Pch)

println("Pdis")
Pdis = value.(model[:Pdis]) # Extracts the DenseAxisArray of values
display(Pdis)

E


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
And data, a 1×3 Matrix{Float64}:
 50.0  58.7244  -0.0

E_ID


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 1×3×2 Array{Float64, 3}:
[:, :, 1] =
 72.99  58.7244  -0.0

[:, :, 2] =
 50.0113  19.7895  -0.0

E_RT


...

2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
And data, a 1×3 Matrix{Float64}:
 62.9349  11.9022  1.04584

Pch


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
And data, a 1×3 Matrix{Float64}:
 0.0  0.0  0.0

Pdis


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
And data, a 1×3 Matrix{Float64}:
 0.0  4.0  11.3071

In [12]:
println("ΔPch_ID")
ΔPch_ID = value.(model[:ΔPch_ID]) # Extracts the DenseAxisArray of values
display(ΔPch_ID)

println("ΔPdis_ID")
ΔPdis_ID = value.(model[:ΔPdis_ID]) # Extracts the DenseAxisArray of values
display(ΔPdis_ID)

ΔPch_ID


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 1×3×2 Array{Float64, 3}:
[:, :, 1] =
 24.2  0.0  0.0

[:, :, 2] =
 0.0118618  0.0  0.0

ΔPdis_ID


3-dimensional DenseAxisArray{Float64,3,...} with index sets:
    Dimension 1, [1]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
And data, a 1×3×2 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  -0.0

[:, :, 2] =
 0.0  36.9881  0.0

In [14]:
println("shed")
shed = value.(model[:shed]) # Extracts the DenseAxisArray of values
display(shed)

shed


4-dimensional DenseAxisArray{Float64,4,...} with index sets:
    Dimension 1, [1, 2, 3, 4, 5]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
    Dimension 4, [1, 2, 3]
And data, a 5×3×2×3 Array{Float64, 4}:
[:, :, 1, 1] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

[:, :, 2, 1] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

[:, :, 1, 2] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

[:, :, 2, 2] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

[:, :, 1, 3] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

[:, :, 2, 3] =
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

# Costs

In [15]:
P      = value.(model[:P])
ΔP_up  = value.(model[:ΔP_up])
ΔP_dn  = value.(model[:ΔP_dn])
rU     = value.(model[:rU])
rD     = value.(model[:rD])

4-dimensional DenseAxisArray{Float64,4,...} with index sets:
    Dimension 1, [1, 2, 3, 4]
    Dimension 2, [1, 2, 3]
    Dimension 3, [1, 2]
    Dimension 4, [1, 2, 3]
And data, a 4×3×2×3 Array{Float64, 4}:
[:, :, 1, 1] =
  0.0  -0.0      0.0
 -0.0  -0.0      0.0
  0.0   0.0      0.0
 20.0  13.6681  12.1

[:, :, 2, 1] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0   0.0
 18.8  18.8  17.1

[:, :, 1, 2] =
  0.0  -0.0       0.0
 -0.0  -0.0       0.0
  0.0   4.19292   4.19292
 20.0  20.0      20.0

[:, :, 2, 2] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0  -0.0
 19.0  19.0  18.4

[:, :, 1, 3] =
 20.0  20.0  20.0
 -0.0  20.0  20.0
 20.0  20.0  20.0
 20.0  20.0  20.0

[:, :, 2, 3] =
  0.0  -0.0  -0.0
  0.0  -0.0   0.0
 -0.0  -0.0  -0.0
 20.0  20.0  20.0

In [16]:
prod(i,t,ω,σ) =
    P[i,t] +
    ΔP_up[i,t,ω] - ΔP_dn[i,t,ω] +
    rU[i,t,ω,σ] - rD[i,t,ω,σ]

prod (generic function with 1 method)

In [17]:
cost_unit_time = Dict()

for i in G, t in T
    expected_cost = 0.0
    for ω in Ω, σ in Σ
        prob = data.πΩ[ω] * data.πΣ[(ω,σ)]
        realized = P[i,t] +
                   ΔP_up[i,t,ω] - ΔP_dn[i,t,ω] +
                   rU[i,t,ω,σ] - rD[i,t,ω,σ]
        expected_cost += prob * data.C_DA[i] * realized
    end
    cost_unit_time[(i,t)] = expected_cost
end

In [20]:
cost_unit_time

Dict{Any, Any} with 12 entries:
  (1, 2) => 2952.48
  (3, 1) => 2673.28
  (1, 3) => 2762.4
  (3, 2) => 2496.48
  (3, 3) => 2192.97
  (4, 1) => 839.337
  (2, 1) => 2000.0
  (4, 2) => 1720.04
  (2, 2) => 1960.4
  (4, 3) => 1195.21
  (2, 3) => 1832.85
  (1, 1) => 2934.02

In [18]:
cost_time = Dict()

for t in T
    cost_time[t] = sum(cost_unit_time[(i,t)] for i in G)
end

In [19]:
cost_time

Dict{Any, Any} with 3 entries:
  2 => 9129.4
  3 => 7983.42
  1 => 8446.64

In [23]:
cost_unit = Dict()

for i in G
    cost_unit[i] = sum(cost_unit_time[(i,t)] for t in T)
end

In [24]:
cost_unit

Dict{Any, Any} with 4 entries:
  4 => 3754.58
  2 => 5793.25
  3 => 7362.73
  1 => 8648.9

# Dual variable results

### E_RT

In [42]:
for t in data.T, ω in data.Ω, σ in data.Σ
    println("t=$t, ω=$ω, σ=$σ : ",
        dual(model[:SOC_RT][1,t,ω,σ]))
end

t=1, ω=1, σ=1 : -5.218754278216385
t=1, ω=1, σ=2 : -15.656262834649155
t=1, ω=1, σ=3 : -2.501052631578947
t=1, ω=2, σ=1 : -7.95991814152196
t=1, ω=2, σ=2 : -23.87975442456588
t=1, ω=2, σ=3 : -7.95991814152196
t=2, ω=1, σ=1 : -5.266800000000003
t=2, ω=1, σ=2 : -12.160269534232162
t=2, ω=1, σ=3 : -4.053423178077387
t=2, ω=2, σ=1 : -8.882573877276434
t=2, ω=2, σ=2 : -26.64772163182931
t=2, ω=2, σ=3 : -8.882573877276437
t=3, ω=1, σ=1 : -5.266800000000002
t=3, ω=1, σ=2 : -5.643000000000003
t=3, ω=1, σ=3 : -0.0
t=3, ω=2, σ=1 : -8.033200000000004
t=3, ω=2, σ=2 : -24.099600000000006
t=3, ω=2, σ=3 : -8.033200000000008


In [49]:
for t in data.T, ω in data.Ω
    val = sum(data.πΣ[(ω,σ)] *
        dual(model[:SOC_RT][1,t,ω,σ])
        for σ in data.Σ)

    println("t=$t, ω=$ω : ", -val)
end

t=1, ω=1 : 10.93771908274856
t=1, ω=2 : 17.51181991134831
t=2, ω=1 : 9.160206356154774
t=2, ω=2 : 19.541662530008157
t=3, ω=1 : 4.439160000000002
t=3, ω=2 : 17.673040000000007


### RT power balance

In [45]:
for n in data.N, t in data.T, ω in data.Ω, σ in data.Σ
    println("n=$n, t=$t, ω=$ω, σ=$σ : ",
        dual(model[:RT_balance][n,t,ω,σ]))
end

n=1, t=1, ω=1, σ=1 : 4.957816564305566
n=1, t=1, ω=1, σ=2 : 14.873449692916692
n=1, t=1, ω=1, σ=3 : 2.3759999999999994
n=1, t=1, ω=2, σ=1 : 7.561922234445862
n=1, t=1, ω=2, σ=2 : 22.68576670333755
n=1, t=1, ω=2, σ=3 : 7.561922234445863
n=1, t=2, ω=1, σ=1 : 5.544
n=1, t=2, ω=1, σ=2 : 12.800283720244375
n=1, t=2, ω=1, σ=3 : 3.8507520191735165
n=1, t=2, ω=2, σ=1 : 9.350077765554136
n=1, t=2, ω=2, σ=2 : 28.050233296662427
n=1, t=2, ω=2, σ=3 : 9.35007776555414
n=1, t=3, ω=1, σ=1 : 5.5440000000000005
n=1, t=3, ω=1, σ=2 : 5.94
n=1, t=3, ω=1, σ=3 : -0.0
n=1, t=3, ω=2, σ=1 : 8.456000000000001
n=1, t=3, ω=2, σ=2 : 25.368000000000002
n=1, t=3, ω=2, σ=3 : 8.456000000000003
n=2, t=1, ω=1, σ=1 : 4.957816564305568
n=2, t=1, ω=1, σ=2 : 14.873449692916695
n=2, t=1, ω=1, σ=3 : 2.376
n=2, t=1, ω=2, σ=1 : 7.561922234445863
n=2, t=1, ω=2, σ=2 : 22.685766703337585
n=2, t=1, ω=2, σ=3 : 7.561922234445865
n=2, t=2, ω=1, σ=1 : 5.544000000000002
n=2, t=2, ω=1, σ=2 : 12.800283720244378
n=2, t=2, ω=1, σ=3 : 3.8507

In [50]:
for n in data.N, t in data.T, ω in data.Ω
    val = sum(data.πΣ[(ω,σ)] *
        dual(model[:RT_balance][n,t,ω,σ])
        for σ in data.Σ)

    println("node=$n, t=$t, ω=$ω : ", val)
end

node=1, t=1, ω=1 : 10.390833128611126
node=1, t=1, ω=2 : 16.636228915780872
node=1, t=2, ω=1 : 9.559120635981328
node=1, t=2, ω=2 : 20.570171084219112
node=1, t=3, ω=1 : 4.6728000000000005
node=1, t=3, ω=2 : 18.6032
node=2, t=1, ω=1 : 10.39083312861113
node=2, t=1, ω=2 : 16.636228915780894
node=2, t=2, ω=1 : 9.559120635981332
node=2, t=2, ω=2 : 20.570171084219115
node=2, t=3, ω=1 : 4.672800000000002
node=2, t=3, ω=2 : 18.603200000000005
node=3, t=1, ω=1 : 10.39083312861113
node=3, t=1, ω=2 : 16.636228915780894
node=3, t=2, ω=1 : 9.559120635981332
node=3, t=2, ω=2 : 20.570171084219115
node=3, t=3, ω=1 : 4.672800000000002
node=3, t=3, ω=2 : 18.603200000000005
node=4, t=1, ω=1 : 10.39083312861113
node=4, t=1, ω=2 : 16.63622891578089
node=4, t=2, ω=1 : 9.55912063598133
node=4, t=2, ω=2 : 20.570171084219112
node=4, t=3, ω=1 : 4.6728000000000005
node=4, t=3, ω=2 : 18.6032
node=5, t=1, ω=1 : 10.39083312861113
node=5, t=1, ω=2 : 16.636228915780894
node=5, t=2, ω=1 : 9.55912063598133
node=5, t=

In [51]:
for t in data.T, ω in data.Ω
    val = sum(data.πΣ[(ω,σ)] *
        dual(model[:RT_balance][2,t,ω,σ])
        for σ in data.Σ)

    println("t=$t, ω=$ω : ", val)
end

t=1, ω=1 : 10.39083312861113
t=1, ω=2 : 16.636228915780894
t=2, ω=1 : 9.559120635981332
t=2, ω=2 : 20.570171084219115
t=3, ω=1 : 4.672800000000002
t=3, ω=2 : 18.603200000000005
